# G-Portal anonymous search

G-Portal's search is **anonymous** — only the SFTP download step needs a free G-Portal account. This notebook queries `gportal.search(...)` for the first matching products of a GCOM-C/SGLI L3 ocean product without ever calling `gportal.download(...)`.

> Needs the `[jaxa]` extra (`pip install 'earthlens[jaxa]'`); no credentials.

## Open the catalog and pick a product

In [ ]:
from earthlens.jaxa import Catalog

cat = Catalog()
row = cat.get('sgli-l380')  # 'sgli-l380' is an alias for 'sgli-l3-nwlr'
print('canonical key:', row.key)
print('protocol:     ', row.protocol)
print('short_name:   ', row.short_name)
print('description:  ', row.description)

## Anonymous search

The `gportal` SDK's `search()` does not require credentials — only the actual `download()` step does. The call below queries the live G-Portal catalogue for products matching a 1-day window.

In [ ]:
import gportal

result = gportal.search(
    dataset_ids=[row.short_name],
    start_time='2024-01-01',
    end_time='2024-01-02',
    count=3,
)
matched = result.matched() or 0
print(f'matched products: {matched}')

## Inspect the first product

In [ ]:
products = list(result.products())
for p in products[:3]:
    print(f'id:        {p.id}')
    print(f'data_path: {p.data_path}')
    print(f'data_url:  {p.data_url[:90] if p.data_url else None}')
    print()

## Downloading the product

The SFTP download step needs a G-Portal account (free at <https://gportal.jaxa.jp/gpr/user/regist1>). Set `$GPORTAL_USERNAME` and `$GPORTAL_PASSWORD` in your environment and use the facade:

```python
from earthlens import EarthLens

lens = EarthLens(
    data_source='jaxa',
    variables=['sgli-l380'],
    start='2024-01-01',
    end='2024-01-02',
    lat_lim=[0.0, 30.0],
    lon_lim=[120.0, 150.0],
    path='./out',
)
written = lens.download()  # SFTP fetch via gportal.download(...)
```

`JaxaAuth.configure()` reads `$GPORTAL_USERNAME` and `$GPORTAL_PASSWORD` and threads them straight to `gportal.download(username=, password=)` so the SDK's module-level credential globals stay untouched between requests.